# Ma Soi — RL (PPO from the BC clone) on Kaggle, headless

Same experiment as section 6 of `train_bc_local.ipynb` — v3 run names and flags (spec
`docs/superpowers/specs/2026-09-17-rl-ppo-from-bc-design.md`, D11). Built for **"Save &
Run All"**: Kaggle runs it in the background, the browser can be closed and your machine
can be off. CPU sessions have **4 cores** and a **12 h** limit, so each version runs ONE
stage and stops itself at 11 h; the next version resumes where it stopped.

## One-time setup

1. **Kaggle account, phone-verified** (Settings → Phone verification). Needed to turn on
   Internet, which `npm ci` and the Node download require.
2. **Package the code** on your machine (only COMMITTED files go in):

   ```powershell
   git archive --format=zip HEAD -o .tmp\repo.zip
   ```

3. **Dataset:** kaggle.com → Datasets → New Dataset → upload `repo.zip` (private).
   Kaggle may unzip it; this notebook handles both forms.
4. **Notebook:** Code → New Notebook → File → Import Notebook → this file.
   Right panel: **Add Input** → your dataset; **Settings**: Accelerator **None (CPU)**,
   **Internet ON**, Persistence off is fine.

## Each run (one stage)

1. Set `STAGE` in the first code cell (start with `"village"`).
2. From the 2nd run on: **Add Input → Your Work → this notebook** → pick the PREVIOUS
   version, so its `rl/` output is restored.
3. **Save Version → Save & Run All (Commit)**. Close the tab if you like.
4. When it finishes, open the version → **Output**/log. The last line says what to do:
   `>>> NEXT: ...` (next stage, or the SAME stage again if the 11 h budget ran out).

Stages: `village` → (`village-lr3` if needed) → `wolves` → (`wolves-lr3`) → `night`
(optional) → `confirm`. Send the `VERDICT` block to Claude; the candidate model is
saved as `candidate.weights.json` in that version's output.

After a code change: re-run `git archive`, upload a new dataset version, and the
notebook uses it automatically (it prints the commit).

In [ ]:
# ======================= EDIT THIS CELL, then "Save Version" → "Save & Run All" =======================
# One stage per version, in this order (the last cell prints which one to run next):
#   "village"      20 iterations, village side                     (~7–9 h)
#   "village-lr3"  only if "village" never promoted
#   "wolves"       20 iterations, wolves side, from the village champion
#   "wolves-lr3"   only if "wolves" never promoted
#   "night"        optional: 10 iterations per side, night only (only if village AND wolves promoted)
#   "confirm"      fresh-seed benchmark of the final candidate + VERDICT (~1 h)
STAGE = "village"

BUDGET_HOURS = 11.0   # Kaggle CPU sessions stop at 12 h; stop cleanly before that so the output is saved

## 0. Code, Node.js, dependencies

In [ ]:
import glob
import json
import os
import shutil
import subprocess
import sys
import tarfile
import time
import urllib.request
import zipfile
from pathlib import Path

T0 = time.time()
DEADLINE = T0 + BUDGET_HOURS * 3600

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")          # saved as this version's output
RL = WORK / "rl"                        # run folders live here (pruned after each iteration)
ROOT = Path("/tmp/repo")                # code + node_modules: NOT saved to output
TRAIN_DIR = ROOT / "ai-training"
NODE_VERSION = "v22.17.0"


def find_code():
    zips = [Path(p) for p in glob.glob(str(INPUT / "**" / "repo.zip"), recursive=True)]
    if zips:
        return "zip", zips[0]
    # Kaggle may auto-extract an uploaded zip: look for the repo root itself.
    for pkg in glob.glob(str(INPUT / "**" / "package.json"), recursive=True):
        root = Path(pkg).parent
        if (root / "ai-training" / "rl_loop.py").exists():
            return "dir", root
    raise SystemExit("repo not found under /kaggle/input - attach the dataset holding repo.zip (Step B)")


kind, src = find_code()
shutil.rmtree(ROOT, ignore_errors=True)
if kind == "zip":
    with zipfile.ZipFile(src) as archive:
        archive.extractall(ROOT)
        commit = archive.comment.decode() or "?"
else:
    shutil.copytree(src, ROOT)
    commit = "? (extracted dataset: no zip comment)"
print("code:", src, "| commit", commit)

# Node 22 from the official tarball (no apt / root needed).
node_dir = Path(f"/tmp/node-{NODE_VERSION}-linux-x64")
if not (node_dir / "bin" / "node").exists():
    url = f"https://nodejs.org/dist/{NODE_VERSION}/node-{NODE_VERSION}-linux-x64.tar.xz"
    try:
        tar_path, _ = urllib.request.urlretrieve(url, "/tmp/node.tar.xz")
    except OSError as error:
        raise SystemExit(f"cannot download Node ({error}). Settings -> Internet must be ON "
                         "(needs a phone-verified Kaggle account).")
    with tarfile.open(tar_path) as archive:
        archive.extractall("/tmp")
os.environ["PATH"] = f"{node_dir / 'bin'}:{os.environ['PATH']}"
print("node", subprocess.run(["node", "--version"], capture_output=True, text=True).stdout.strip(),
      "| vCPU", os.cpu_count(), "| python", sys.version.split()[0])

done = subprocess.run("npm ci --no-audit --no-fund --loglevel=error && npm run build:deps --silent",
                      shell=True, cwd=ROOT)
if done.returncode != 0:
    raise SystemExit(f"npm ci / build failed (exit {done.returncode})")
import numpy
import torch
print("deps OK | torch", torch.__version__, "| numpy", numpy.__version__)

## 1. Restore the previous version's runs

In [ ]:
# Resume: "Add Input" -> "Your Work" -> this notebook's previous version. Its output
# (the rl/ folder) is copied back so rl_loop skips every finished step.
RL.mkdir(parents=True, exist_ok=True)
previous = sorted({Path(p).parent for p in glob.glob(str(INPUT / "**" / "rl" / "*" / "state.json"), recursive=True)})
for run in previous:
    dest = RL / run.name
    if not dest.exists():
        shutil.copytree(run, dest)
        print("restored", run.name, "from", run)
for log in glob.glob(str(INPUT / "**" / "rl" / "*.log"), recursive=True):
    if not (RL / Path(log).name).exists():
        shutil.copy2(log, RL / Path(log).name)
print("runs present:", sorted(p.name for p in RL.iterdir()) or "none (fresh start)")

## 2. Helpers

Same run folders and flags as the local/Colab notebooks, plus an 11 h stop so the version ends before Kaggle's 12 h limit.

In [ ]:
NPM = shutil.which("npm") or "npm"
CHAMPION0 = ROOT / "apps" / "server" / "assets" / "models" / "village-bc-0002.weights.json"
NO_NIGHT = "vote,final_vote,hunter_shot"
COMMON = [
    "--temperature", "1", "--lr", "1e-4", "--target-kl", "0.01",
    "--shaping-alpha", "1", "--baseline", "role", "--bench-every", "5",
]
SIDE_STAGE = ["--bench-every", "10", "--balance-slack", "-1"]   # spec D11 (later flags win)
RL_ENV = {**os.environ, "PYTHONUTF8": "1", "PYTHONIOENCODING": "utf-8"}
sys.path.insert(0, str(TRAIN_DIR))


class OutOfTime(Exception):
    pass


def run_logged(cmd, log, cwd=TRAIN_DIR):
    # Stream output here + into a log. Stops the child at DEADLINE so the version ends
    # normally and Kaggle saves the output; rl_loop resumes from its .done markers.
    log.parent.mkdir(parents=True, exist_ok=True)
    start = time.time()
    with log.open("a", encoding="utf8") as f:
        f.write(f"\n$ {' '.join(map(str, cmd))}\n")
        proc = subprocess.Popen([str(c) for c in cmd], cwd=str(cwd), env=RL_ENV, stdout=subprocess.PIPE,
                                stderr=subprocess.STDOUT, text=True, encoding="utf8", errors="replace",
                                start_new_session=True)
        for line in proc.stdout:
            print(line, end="")
            f.write(line)
            if time.time() > DEADLINE:
                os.killpg(proc.pid, 15)   # whole group: npm/tsx children too
                proc.wait()
                f.write("\n[stopped at the time budget]\n")
                raise OutOfTime()
        proc.wait()
    print(f"\n{time.time() - start:.0f}s | exit {proc.returncode}")
    if proc.returncode != 0:
        raise SystemExit(f"failed (exit {proc.returncode}) - full log: {log}")


def rl_loop(name, champion, side, iterations, games, extra=()):
    out = RL / name
    run_logged([sys.executable, "rl_loop.py", "--champion", champion, "--side", side,
                "--iterations", iterations, "--games", games, "--out", out, *COMMON, *extra],
               RL / f"{name}.log")
    return out


def latest_champion(out):
    files = sorted((out / "champions").glob("champion-*.weights.json"))
    return (files[-1], len(files) > 1) if files else (None, False)


def show_state(out):
    state = json.loads((out / "state.json").read_text(encoding="utf8"))
    print(f"{'iter':>4} {'score':>7} {'confirm':>8} {'other':>7} {'imbal':>6}")
    for row in state["scores"]:
        print(f"{row['iteration']:>4} {row['score']:>+7.2f} {row.get('confirmScore', float('nan')):>+8.2f} "
              f"{row.get('otherSide', float('nan')):>+7.2f} {row.get('imbalance', float('nan')):>6.2f}")
    champion, promoted = latest_champion(out)
    print(f"champion score {state['championScore']:+.2f} | latest champion {champion.name} |",
          "PROMOTED" if promoted else "never promoted")
    return champion, promoted


def promoted_champion(*names):
    # Latest promoted champion among the given runs (first name wins), or None.
    for name in names:
        out = RL / name
        if (out / "champions").exists():
            champion, promoted = latest_champion(out)
            if promoted:
                return champion
    return None


def candidate():
    # Final model to confirm: the most advanced stage that promoted.
    return (promoted_champion("bc-night-wolves-v3", "bc-night-village-v3")
            or promoted_champion("bc-wolves-v3-lr3", "bc-wolves-v3")
            or promoted_champion("bc-village-v3-lr3", "bc-village-v3"))


print("helpers ready | stage:", STAGE, f"| budget {BUDGET_HOURS} h")

## 3. Prerequisites

In [ ]:
loop_help = subprocess.run([sys.executable, "rl_loop.py", "--help"], cwd=str(TRAIN_DIR), env=RL_ENV,
                           capture_output=True, text=True, encoding="utf8").stdout
wanted = ("--learned-decisions", "--balance-slack", "--other-side-slack", "--anchor-kl", "--prune-rollouts")
missing = [flag for flag in wanted if flag not in loop_help]
if missing:
    raise SystemExit(f"missing {missing} - repo.zip is from an older commit; re-run git archive")
if not CHAMPION0.exists():
    raise SystemExit(f"missing {CHAMPION0}")
print("OK - pipeline has the four-decision flag, both gates, the KL anchor and pruning")

## 4. Run the stage

The last lines always print `>>> NEXT:`.

In [ ]:
STAGES = {
    "village": ("bc-village-v3", "village", 20, ["--train-decisions", NO_NIGHT, *SIDE_STAGE]),
    "village-lr3": ("bc-village-v3-lr3", "village", 20, ["--train-decisions", NO_NIGHT, *SIDE_STAGE, "--lr", "3e-4"]),
    "wolves": ("bc-wolves-v3", "wolves", 20,
               ["--train-decisions", NO_NIGHT, "--shaping-decisions", "vote", *SIDE_STAGE]),
    "wolves-lr3": ("bc-wolves-v3-lr3", "wolves", 20,
                   ["--train-decisions", NO_NIGHT, "--shaping-decisions", "vote", *SIDE_STAGE, "--lr", "3e-4"]),
}


def start_model(stage):
    if stage.startswith("village"):
        return CHAMPION0
    champion = promoted_champion("bc-village-v3-lr3", "bc-village-v3")
    if champion is None:
        raise SystemExit("wolves needs a PROMOTED village champion - run 'village' (then 'village-lr3') first, "
                         "and attach that version's output as input")
    return champion


next_hint = None
try:
    if STAGE in STAGES:
        name, side, iterations, extra = STAGES[STAGE]
        out = rl_loop(name, start_model(STAGE), side, iterations, 3000, extra)
        _, ok = show_state(out)
        if STAGE.startswith("village"):
            next_hint = "wolves" if ok else ("village-lr3" if STAGE == "village" else
                                             "STOP: village never promoted -> send show_state to Claude (sub-project B)")
        else:
            next_hint = "night (optional) or confirm" if ok else ("wolves-lr3" if STAGE == "wolves" else "confirm")
    elif STAGE == "night":
        base = candidate()
        if not (promoted_champion("bc-village-v3-lr3", "bc-village-v3")
                and promoted_champion("bc-wolves-v3-lr3", "bc-wolves-v3")):
            raise SystemExit("night needs BOTH village and wolves promoted - skip to 'confirm'")
        out = rl_loop("bc-night-village-v3", base, "village", 10, 3000, ["--train-decisions", "night", *SIDE_STAGE])
        show_state(out)
        out = rl_loop("bc-night-wolves-v3", candidate(), "wolves", 10, 3000,
                      ["--train-decisions", "night", "--shaping-decisions", "vote", *SIDE_STAGE])
        show_state(out)
        next_hint = "confirm"
    elif STAGE == "confirm":
        CANDIDATE = candidate()
        if CANDIDATE is None:
            raise SystemExit("no promoted champion in any run - nothing to confirm")
        BENCH = ["--games", "300", "--repeat", "5", "--seed", "confirm-0917",
                 "--setups", "baseline,village,wolves,all,teacher",
                 "--learned-decisions", "vote,night,final,hunter"]
        BASE_JSON = RL / "confirm-v3-bc0002.json"
        CAND_JSON = RL / "confirm-v3-candidate.json"
        for model, dest in ((CHAMPION0, BASE_JSON), (CANDIDATE, CAND_JSON)):
            if not dest.exists():
                run_logged([NPM, "run", "ai:benchmark", "--", "--model", model, *BENCH, "--out", dest],
                           RL / "confirm.log", cwd=ROOT)
        from rl_loop import imbalance_of, score_of
        promoted_sides = {"village"} | ({"wolves"} if promoted_champion("bc-wolves-v3-lr3", "bc-wolves-v3") else set())
        delta = {s: score_of(CAND_JSON, s) - score_of(BASE_JSON, s) for s in ("village", "wolves")}
        imbalance = imbalance_of(CAND_JSON)
        violations = sum(r["violations"] for r in json.loads(CAND_JSON.read_text(encoding="utf8"))["rows"])
        checks = {
            "promoted sides >= +2": all(delta[s] >= 2.0 for s in promoted_sides),
            "no side below -1": all(d >= -1.0 for d in delta.values()),
            "imbalance <= 6.8": imbalance <= 6.8,
            "zero violations": violations == 0,
        }
        print("=" * 60)
        for side, d in delta.items():
            print(f"{side:<8} delta vs bc-0002 {d:+.2f}" + ("  (trained)" if side in promoted_sides else ""))
        print(f"imbalance {imbalance:.2f} | violations {violations}")
        for check, ok in checks.items():
            print(f"  [{'x' if ok else ' '}] {check}")
        print("VERDICT:", "PASS" if all(checks.values()) else "FAIL")
        print("candidate:", CANDIDATE)
        print("=" * 60)
        shutil.copy2(CANDIDATE, WORK / "candidate.weights.json")
        next_hint = "send the VERDICT block to Claude; download candidate.weights.json from the output"
    else:
        raise SystemExit(f"unknown STAGE {STAGE!r}")
except OutOfTime:
    next_hint = (f"SAME STAGE AGAIN ({STAGE!r}): the time budget ran out. New version: attach THIS version's "
                 "output as input, keep STAGE, Save & Run All - finished iterations are skipped.")

print("\n>>> NEXT:", next_hint)
print(f">>> elapsed {(time.time() - T0) / 3600:.1f} h")